# Predicción de Demanda y Sistema Inteligente de Alertas de Inventario

## Planteamiento del Problema
En la gestión diaria de un depósito o almacén, los conteos de inventario "a ciegas" o preventivos consumen tiempo y recursos valiosos. Además, el desfasaje constante entre el stock del sistema y la realidad física (debido a ventas en tiempo real, mermas o errores) dificulta la toma de decisiones al momento de realizar pedidos a proveedores.

## Objetivo del Proyecto
El objetivo principal es analizar un dataset histórico de ventas para entrenar un modelo de Machine Learning (Scikit-Learn) capaz de predecir la demanda futura. El fin último es utilizar estas predicciones para simular un sistema de alertas que indique qué productos específicos requieren un conteo físico, optimizando así el tiempo del personal.

## Fases del Proyecto a evaluar:
1. **Análisis Exploratorio (EDA):** Identificar patrones de estacionalidad, tendencias de ventas y comparar el volumen de salida entre distintas tiendas y artículos.
2. **Ingeniería de Características:** Descomponer variables temporales (fechas) para alimentar al algoritmo predictivo.
3. **Modelado y Predicción:** Entrenar un modelo de regresión para pronosticar las ventas a corto plazo.
4. **Lógica de Negocio (Alertas):** Establecer la lógica para detectar discrepancias entre la demanda proyectada y el stock simulado.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocesamiento

In [2]:
df = pd.read_csv('train.csv')

print('--- INFORMACIÓN DEL DATASET ---')
df.info()

print('\n--- PRIMERAS 5 FILAS ---')
display(df.head())

--- INFORMACIÓN DEL DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555362 entries, 0 to 555361
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    555362 non-null  object 
 1   store   555362 non-null  int64  
 2   item    555362 non-null  int64  
 3   sales   555361 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 16.9+ MB

--- PRIMERAS 5 FILAS ---


,date,store,item,sales
0,2013-01-01,1,1,13.0
1,2013-01-02,1,1,11.0
2,2013-01-03,1,1,14.0
3,2013-01-04,1,1,13.0
4,2013-01-05,1,1,10.0


Se transforma la columna "date" a tipo *datetime*. Si Python lee la columna como tipo *object* la va a leer como texto plano, lo cual sería un inconveniente si se decidiera acomodar las fechas de cierta manera o calcular de manera rápida si un producto no se estuvo vendiendo.

In [3]:
df['date'] = pd.to_datetime(df['date'])

In [4]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

display(df.head())

,date,store,item,sales,year,month,day,day_of_week
0,2013-01-01,1,1,13.0,2013,1,1,1
1,2013-01-02,1,1,11.0,2013,1,2,2
2,2013-01-03,1,1,14.0,2013,1,3,3
3,2013-01-04,1,1,13.0,2013,1,4,4
4,2013-01-05,1,1,10.0,2013,1,5,5


## 2. Análisis Exploratio de Datos (EDA)

### 2.1. Volumen de Ventas Totales por Tienda
Antes de predecir el futuro, se necesita entender el volumen histórico. El objetivo de este gráfico es identificar si todas las sucursales manejan un volumen de demanda similar o si existen "tiendas principales" que requerirán mayor atención de stock.

In [5]:
ventas_por_tienda = df.groupby('store')['sales'].sum().reset_index()

ventas_por_tienda['store'] = ventas_por_tienda['store'].astype(str)

fig_tiendas = px.bar(
    ventas_por_tienda,
    template = 'plotly_dark',
    x = 'store',
    y = 'sales',
    title = '📦 Volumen Histórico de Ventas Totales por Tienda 🏬',
    labels = {'store': 'Número de Tienda 🏬', 'sales': 'Ventas Totales 🛒​'},
    color = 'sales',
    color_continuous_scale = 'Teal'
)

fig_tiendas.update_traces(
    hovertemplate = 'Número de Tienda 🏬=%{x}<br>Ventas Totales 🛒=%{y:,.0f}<extra></extra>'
)

fig_tiendas.update_layout(
    separators = ',.',
    title_x = 0.5,
    title_font = dict(size = 25),
    xaxis_title_font = dict(size = 20),
    yaxis_title_font = dict(size = 20),
    yaxis = dict(ticksuffix = ' u.')
)

fig_tiendas.show()

### 2.2. Análisis Temporal y Estacionalidad
El objetivo de este paso es entender cómo evolucionan las unidades vendidas a lo largo del tiempo. Se busca identificar dos factores clave para el inventario:
1. **Tendencia:** ¿El negocio está creciendo o achicandose con los años?
2. **Estacionalidad:** ¿Hay picos de demanda que se repiten en las mismas fechas todos los años?

In [6]:
ventas_por_fecha = df.groupby('date')['sales'].sum().reset_index()

fig_temporal = px.line(
    ventas_por_fecha,
    template = 'plotly_dark',
    x = 'date',
    y = 'sales',
    title = '📈 Evolución de Unidades Vendidas a lo largo del Tiempo',
    labels = {'date': 'Fecha 📅', 'sales': 'Unidades Vendidas 📦'}
)

fig_temporal.update_traces(
    line = dict(color = '#00CC96', width = 2),
    hovertemplate = 'Fecha 📅=%{x}<br>Unidades 📦=%{y:,.0f}<extra></extra>'
)

fig_temporal.update_layout(
    separators = ',.',
    title_x = 0.5,
    title_font = dict(size =25),
    xaxis_title_font = dict(size = 20),
    yaxis_title_font = dict(size = 20),
    yaxis = dict(ticksuffix = ' u.')
)

fig_temporal.show()

### 2.2.1. Comportamiento estacional por sucursal
Al detectar picos recurrentes en las ventas totales durante los meses de julio, se cruza la línea de tiempo con la variable de tiendas ('store'). El objetivo es verificar si este comportamiento es un patrón generalizado de la empresa o si está impulsado únicamente por sucursales específicas.

In [7]:
ventas_fecha_tienda = df.groupby(['date', 'store'])['sales'].sum().reset_index()

ventas_fecha_tienda['store'] = ventas_fecha_tienda['store'].astype(str)

fig_temporal_tiendas = px.line(
    ventas_fecha_tienda,
    template = 'plotly_dark',
    x = 'date',
    y = 'sales',
    color = 'store',
    title = '📈 Evolución Temporal discriminada por Tienda 🏬',
    labels = {'date': 'Fecha 📅', 'sales': 'Unidades 📦', 'store': 'Tienda'}
)

fig_temporal_tiendas.update_traces(
    hovertemplate = 'Fecha 📅=%{x}<br>Unidades 📦=%{y:,.0f}<extra></extra>'
)

fig_temporal_tiendas.update_layout(
    separators = ',.',
    title_x = 0.5,
    title_font = dict(size = 25),
    xaxis_title_font = dict(size = 20),
    yaxis_title_font = dict(size = 20),
    yaxis = dict(ticksuffix = ' u.')
)

fig_temporal_tiendas.show()

### 2.2.2. Ventas por Día de la Semana
Este gráfico desagrega las ventas según el día de la semana. Esto le va a servir al modelo predictivo para entender la dinámica diaria del negocio y determinar exactamente en qué días se necesita mayor volumen de mercadería (y qué días son ideales para reponer o recontar stock).


In [8]:
ventas_dias_semana = df.groupby('day_of_week')['sales'].sum().reset_index()

nombres_dias = {
    0: 'Lunes',
    1: 'Martes',
    2: 'Miércoles',
    3: 'Jueves',
    4: 'Viernes',
    5: 'Sábado',
    6: 'Domingo'
}
ventas_dias_semana['day_of_week'] = ventas_dias_semana['day_of_week'].map(nombres_dias)

fig_dias = px.bar(
    ventas_dias_semana,
    template = 'plotly_dark',
    x = 'day_of_week',
    y = 'sales',
    title = '📊 Volumen de Unidades Vendidas por Día de la Semana',
    labels = {'day_of_week': 'Día de la Semana 🗓️', 'sales': 'Unidades Vendidas 📦'},
    color = 'sales',
    color_continuous_scale = 'Teal'
)

fig_dias.update_traces(
    hovertemplate = 'Día 🗓️=%{x}<br>Unidades 📦=%{y:,.0f}<extra></extra>'
)

fig_dias.update_layout(
    separators = ',.',
    title_x = 0.5,
    title_font = dict(size = 25),
    xaxis_title_font = dict(size = 20),
    yaxis_title_font = dict(size = 20),
    yaxis = dict(ticksuffix = ' u.')
)

fig_dias.show()

## 3. Preparación de Datos para Machine Learning

### 3.1. Separación de Variables (X e y)
Para entrenar al modelo predictivo, se necesita dividir el conjunto de datos en dos partes:
* **La variable 'y' (El Objetivo):** Es la columna que se quiere que el modelo aprenda a adivinar (las ventas).
* **La variable 'X' (Las Pistas o Features):** Son todas las demás variables que el modelo usará para encontrar patrones. Acá se elimina la columna objetivo y la columna de fecha original (ya que el modelo usará las columnas numéricas de año, mes y día que creamos en el Feature Engineering).

In [9]:
y = df['sales']

X = df.drop(columns = ['sales', 'date'])

print('Las columnas que el modelo usará para predecir (X) son: ')
print(X.columns.tolist())

Las columnas que el modelo usará para predecir (X) son: 
['store', 'item', 'year', 'month', 'day', 'day_of_week']


### 3.2. División de Datos de Entrenamiento y Prueba

Los datos se separan en dos bloques:
* **Train (80%):** Datos del pasado para que el modelo aprenda los patrones (tendencia y estacionalidad).
* **Test (20%):** Datos del futuro (los últimos meses registrados) para poner a prueba el modelo y evaluar su precisión.

**Nota técnica:** Al tratarse de datos temporales, se desactiva la mezcla aleatoria ('shuffle = False') para evitar predecir el pasado usando el futuro.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, shuffle = False)

print(f'📚 Datos para ESTUDIAR (Train): {X_train.shape[0]:,.0f} filas')
print(f'📝 Datos para el EXAMEN (Test): {X_test.shape[0]:,.0f} filas')

📚 Datos para ESTUDIAR (Train): 444,289 filas
📝 Datos para el EXAMEN (Test): 111,073 filas


## 4. Entrenamiento del Modelo de Machine Learning

Para este proyecto se utilizará un **Random Forest Regressor** (Bosque Aleatorio). Este algoritmo funciona creando múltiples "árboles de decisión" que analizan las variables (como la sucursal, el artículo, la fecha exacta y el día de la semana) desde distintos ángulos. Al promediar las conclusiones de todos los árboles, se logran predicciones mucho más estables y precisas que si se usara un solo modelo simple.

In [11]:
from sklearn.ensemble import RandomForestRegressor

modelo_rf = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42,
    n_jobs = -1
)

print('🧠 Entrenando el modelo... (esto puede tardar unos segundos o minutos) ⏳')
modelo_rf.fit(X_train, y_train)

print('✅ ¡Modelo entrenado exitosamente! El algoritmo ya entendió el negocio.')

🧠 Entrenando el modelo... (esto puede tardar unos segundos o minutos) ⏳
✅ ¡Modelo entrenado exitosamente! El algoritmo ya entendió el negocio.
